## 🛠️ Pipeline de Dados SaaS - Parte 1: Geração de Dados Sintéticos

**Objetivo:** Construir a camada de dados bruta simulando o banco de dados transacional de uma empresa B2B SaaS.  
**Tecnologias:** Python, Pandas, Faker.

> **Contexto de Negócio:** Para alimentar o nosso *Data Warehouse* e validar as métricas do painel executivo, precisamos de um histórico realista de operações. Este script gera 3 anos de dados cruzados de clientes, planos de assinatura e pagamentos. Para testarmos o motor de análise do Power BI, injetamos intencionalmente ruídos do mundo real, como cancelamentos (*Churn*), atrasos e falhas de pagamento (*Inadimplência*).

In [11]:
!pip install pandas faker


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [ ]:
fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)

print("Ambiente de desenvolvimento configurado!")

Environment setup complete. Ready to build the English Standard Data Factory!


## 1. Tabela Dimensão: Planos de Assinatura (`dim_plans`)
Definição do portfólio de produtos da empresa. O modelo de precificação é baseado em três *tiers* (Basic, Pro, Enterprise), refletindo a estrutura clássica de receita escalável em SaaS.

In [ ]:
print("criando dim_plans")

plans_data = [
    {'plan_id': 1, 'plan_name': 'Basic', 'monthly_price': 99.90},
    {'plan_id': 2, 'plan_name': 'Pro', 'monthly_price': 299.90},
    {'plan_id': 3, 'plan_name': 'Enterprise', 'monthly_price': 999.90}
]

df_plans = pd.DataFrame(plans_data)
df_plans.to_csv('dim_plans.csv', index=False)

print("dim_plans.csv criado!")
display(df_plans)

generating dim_plans
dim_plans.csv generated!


,plan_id,plan_name,monthly_price
0,1,Basic,99.9
1,2,Pro,299.9
2,3,Enterprise,999.9


## 2. Tabela Dimensão: Calendário (`dim_calendar`)
Para análises de *Time Intelligence* precisas no Power BI (como crescimento mês a mês ou sazonalidade), é fundamental criarmos uma tabela de dimensão de datas unificada, cobrindo o período de 2023 a 2026.

In [ ]:
print("criando dim_calendar...")

start_date = datetime(2023, 1, 1)
end_date = datetime(2026, 12, 31)
total_days = (end_date - start_date).days + 1

date_list = [start_date + timedelta(days=x) for x in range(total_days)]

calendar_data = []
for d in date_list:
    calendar_data.append({
        'calendar_date': d.strftime('%Y-%m-%d'),
        'year': d.year,
        'month': d.strftime('%B'),
        'quarter': (d.month - 1) // 3 + 1,
        'day_of_week': d.strftime('%A')
    })

df_calendar = pd.DataFrame(calendar_data)
df_calendar.to_csv('dim_calendar.csv', index=False)

print(f"dim_calendar.csv criado com {len(df_calendar)} dias!")
display(df_calendar.head())


generating dim_calendar...
dim_calendar.csv generated with 1461 days!


,calendar_date,year,month,quarter,day_of_week
0,2023-01-01,2023,January,1,Sunday
1,2023-01-02,2023,January,1,Monday
2,2023-01-03,2023,January,1,Tuesday
3,2023-01-04,2023,January,1,Wednesday
4,2023-01-05,2023,January,1,Thursday


## 3. Tabela Dimensão: Clientes B2B (`dim_customers`)
Utilizando a biblioteca `Faker` (com *seed* para reprodutibilidade), simulamos 500 empresas clientes. 
> **Nota de Modelagem:** A inclusão de categorias como `industry` (Setor) e `company_size` (Porte) é crucial para permitirmos análises de Perfil de Cliente Ideal (ICP) na camada de visualização.

In [ ]:
print("Criando dim_customers...")

industries = ['Tecnologia', 'Varejo', 'Saúde', 'Finanças', 'Educação', 'Indústria']
company_sizes = ['Pequena (1-50)', 'Média (51-200)', 'Grande (201+)']

customers_data = []
for i in range(1, 501):
    customers_data.append({
        'customer_id': i,
        'tax_id': fake.cnpj(), # Mantendo CNPJ como tax_id
        'company_name': fake.company(),
        'industry': random.choice(industries),
        'state': fake.state_abbr(),
        'company_size': random.choice(company_sizes)
    })

df_customers = pd.DataFrame(customers_data)
df_customers.to_csv('dim_customers.csv', index=False)

print(f"dim_customers.csv criado com {len(df_customers)} empresas!")
display(df_customers.head())

⏳ Generating dim_customers...
dim_customers.csv generated with 500 companies!


,customer_id,tax_id,company_name,industry,state,company_size
0,1,10.496.582/0001-00,Pastor - ME,Indústria,AL,Pequena (1-50)
1,2,01.384.592/0001-55,Silva Rezende e Filhos,Tecnologia,PI,Grande (201+)
2,3,63.742.015/0001-45,Fernandes,Saúde,GO,Pequena (1-50)
3,4,23.506.871/0001-87,Pereira,Varejo,GO,Pequena (1-50)
4,5,07.139.426/0001-70,Oliveira,Indústria,DF,Pequena (1-50)


## 4. Tabelas Fato: Assinaturas e Pagamentos Financeiros
Este é o "coração" transacional do simulador. Para cada cliente gerado, o algoritmo define aleatoriamente um plano e uma data de entrada.

**Regras de Negócio Implementadas:**
* **Taxa de Churn:** ~35% dos clientes cancelam a assinatura em um período de 1 a 18 meses.
* **Saúde Financeira (Pagamentos):** Para simular um fluxo de caixa realista, os status de pagamento são ponderados:
  - `85%`: Pagos no prazo (com pequena variação de até 5 dias).
  - `10%`: Atrasados.
  - `5%`: Falharam (Inadimplência severa).

In [ ]:
print("Criando assinaturas e pagamentos...")

subscriptions_data = []
payments_data = []
global_payment_id = 1

for index, row in df_customers.iterrows():
    subscription_id = index + 1
    customer_id = row['customer_id']
    
    # Randomly assign a plan
    plan_id = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
    monthly_price = df_plans.loc[df_plans['plan_id'] == plan_id, 'monthly_price'].values[0]
    
    # 1. SUBSCRIPTION LOGIC
    random_days = random.randint(0, 900)
    sub_start_date = start_date + timedelta(days=random_days)
    
    will_churn = random.random() < 0.35
    
    if will_churn:
        active_months = random.randint(1, 18)
        sub_cancel_date = sub_start_date + relativedelta(months=active_months)
        if sub_cancel_date > end_date:
            sub_cancel_date = end_date
        sub_status = 'Cancelada'
    else:
        sub_cancel_date = None
        sub_status = 'Ativa'
        
    subscriptions_data.append({
        'subscription_id': subscription_id,
        'customer_id': customer_id,
        'plan_id': plan_id,
        'start_date': sub_start_date.strftime('%Y-%m-%d'),
        'cancel_date': sub_cancel_date.strftime('%Y-%m-%d') if sub_cancel_date else None,
        'subscription_status': sub_status
    })
    
    # 2. PAYMENTS LOGIC
    current_due_date = sub_start_date
    payment_limit_date = sub_cancel_date if sub_cancel_date else end_date
    
    while current_due_date <= payment_limit_date:
        payment_status = random.choices(['Pago', 'Atrasado', 'Falhou'], weights=[0.85, 0.10, 0.05])[0]
        
        if payment_status == 'Pago':
            delay_days = random.randint(0, 5)
            actual_payment_date = current_due_date + timedelta(days=delay_days)
            payment_date_str = actual_payment_date.strftime('%Y-%m-%d')
        else:
            payment_date_str = None
            
        payments_data.append({
            'payment_id': global_payment_id,
            'subscription_id': subscription_id,
            'due_date': current_due_date.strftime('%Y-%m-%d'),
            'payment_date': payment_date_str,
            'payment_status': payment_status,
            'payment_method': random.choices(['Cartão de Crédito', 'Boleto', 'Pix'], weights=[0.7, 0.2, 0.1])[0],
            'amount_paid': round(monthly_price, 2)
        })
        
        global_payment_id += 1
        current_due_date += relativedelta(months=1)

# Exporting final dataframes
df_subscriptions = pd.DataFrame(subscriptions_data)
df_subscriptions.to_csv('fact_subscriptions.csv', index=False)
print(f"✅ fact_subscriptions.csv gerado com {len(df_subscriptions)} assinaturas!")

df_payments = pd.DataFrame(payments_data)
df_payments.to_csv('fact_payments.csv', index=False)
print(f"fact_payments.csv criado com {len(df_payments)} transações!")

print("fabricação dos dados finalizada!")

⏳ Generating Subscriptions and Payments...
✅ fact_subscriptions.csv generated with 500 records!
✅ fact_payments.csv generated with 8398 transactions!

🚀 Refactored Data Factory execution completed successfully!
